## Import Dependencies


In [1]:
import openai
import instructor
from qdrant_client import QdrantClient
from pydantic import BaseModel,Field

c:\Users\jaysi\Desktop\Desktop\Ai-engineering\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Mock example

In [2]:
prompt="""
You are a helpful assistant
Return an answer to The Question,
Question: Whats is your Name?
"""

In [3]:
response=openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role":"system","content":prompt}
    ]
    ,
    temperature=1
)

In [4]:
print(response.choices[0].message.content)

I am an AI language model created by OpenAI, and I don't have a personal name. You can call me ChatGPT. How can I assist you today?


### Add Instructor (Structured outPut)

In [5]:
client=instructor.from_openai(openai.OpenAI())


In [6]:
class RAGGenerationResponse(BaseModel):
    answer:str=Field(description="The Answer of the Question")

In [7]:
response,raw_response=client.chat.completions.create_with_completion(
    model="gpt-4.1-mini",
    messages=[
        {"role":"system","content":prompt}
    ]
    ,
    temperature=1,
    response_model=RAGGenerationResponse
)

In [15]:
print(type(raw_response.usage))

<class 'openai.types.completion_usage.CompletionUsage'>


In [17]:
raw_response.usage

CompletionUsage(completion_tokens=17, prompt_tokens=95, total_tokens=112, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))

In [18]:
class RAGGenerationResponse(BaseModel):
    anwser:str=Field(description="The Answer of the Question")
    reasoning:str=Field(description="The Reasoning Of the answer")

In [19]:
response,raw_response=client.chat.completions.create_with_completion(
    model="gpt-4.1-mini",
    messages=[
        {"role":"system","content":prompt}
    ]
    ,
    temperature=1,
    response_model=RAGGenerationResponse
)

In [20]:
response

RAGGenerationResponse(anwser='My name is ChatGPT.', reasoning='I am an AI language model created by OpenAI, and I am known as ChatGPT.')

### Rag Example

In [36]:
class RAGGenerationResponse(BaseModel):
    answer:str=Field(description="The Answer of the Question")

In [41]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import Distance,VectorParams,PointStruct
load_dotenv()






# Qdrant host: "http://qdrant:6333" inside docker-compose, "http://localhost:6333" locally.
QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")

def get_embedding(text,model="text-embedding-3-small"):
    response=client.embeddings.create(
        model=model,
        input=text
    )
    
    return response.data[0].embedding

def reteriver_data(query, qdrant_client, k):
    query_embedding = get_embedding(query)

    result = qdrant_client.query_points(
        collection_name="Amazon_items_collection-00",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_rating = []
    retrieved_image_urls = []
    retrieved_prices = []

    for point in result.points:
        payload = point.payload or {}
        retrieved_context_ids.append(payload.get("parent_asin"))
        retrieved_context.append(payload.get("description", ""))
        retrieved_context_rating.append(payload.get("average_rating"))
        similarity_scores.append(point.score)
        retrieved_image_urls.append(payload.get("image_url", ""))
        retrieved_prices.append(payload.get("price"))

    return (
        retrieved_context,
        retrieved_context_ids,
        retrieved_context_rating,
        similarity_scores,
        retrieved_image_urls,
        retrieved_prices
    )


def process_context(context, ids, ratings, scores):
    formatted_context = ""

    for id, chunk, rating, score in zip(ids, context, ratings, scores):
        formatted_context += (
            f"- ID: {id}\n"
            f"  Description: {chunk}\n"
            f"  Rating: {rating}\n"
            f"  Similarity Score: {score:.4f}\n\n"
        )

    return formatted_context

def build_prompt(processed_context, question):
    prompt = f"""
You are a helpful AI shopping assistant.

Use ONLY the information provided in the retrieved product context below to answer the user's question.

Rules:
- Do not make up information.
- If the answer is not present in the context, reply:
  "I couldn't find that information in the retrieved products."
- Keep your answer concise and helpful.
- Mention product IDs when relevant.

======================
Retrieved Context:
{processed_context}
======================

User Question:
{question}

Answer:
"""

    return prompt

def generate_chat(prompt):
    response ,raw_response= client.chat.completions.create_with_completion(
        model="gpt-4.1-mini",
        messages=[
            {
                "role": "system",
                "content": prompt
            },
            
        ],
        temperature=0.2,
        response_model=RAGGenerationResponse
    )

    return response,raw_response


def rag_pipeline(question, top_k=5):
    qdrant_client = QdrantClient(QDRANT_URL)

    retrieved_context = reteriver_data(
        question,
        qdrant_client,
        top_k
    )
   
    processed_context = process_context(
        retrieved_context[0],
        retrieved_context[1],
        retrieved_context[2],
        retrieved_context[3]
    )

    prompt = build_prompt(
        processed_context,
        question
    )
    
    answer ,raw_response= generate_chat(prompt)
    final_result={
        "detamodel":answer,
        "answer":answer,
        "Question":question,
        "raw_response":raw_response,
        "reterived_context_ids":retrieved_context[1],
        "reterived_context":retrieved_context[0],
        "similaritry_ecore":retrieved_context[3]
    }
    return final_result


    



In [ ]:
output=rag_pipeline("can i get some charging Cords");


dict

In [45]:
output

{'detamodel': RAGGenerationResponse(answer='You can get charging cords like:\n- 5-pack iPhone Charger Lightning Cables (3/3/6/6/10 ft) with Apple MFi certification, braided and multi-color (ID: B0BPLX388R).\n- 2-pack 4ft Multi Fast Charging Cable 3 in 1 (Type C/Lightning/Micro USB) with 4.5A output (ID: B09X73T9DQ).\n- 4 in 1 USB C Cable Lightning Cable 45W Flat Braided with Velcro, 3ft length (ID: B0CCNXLP57).\n- 2 Pack USB C to Lightning Cable, 6 ft, MFi Certified, fast charging (ID: B0BV6PWVCG).\n- 3 Pack 10FT Long USB C to Lightning Fast Charging Cables with 20W PD charger blocks (ID: B0CFG7V2QX).\nLet me know if you want details or to purchase any of these.'),
 'answer': RAGGenerationResponse(answer='You can get charging cords like:\n- 5-pack iPhone Charger Lightning Cables (3/3/6/6/10 ft) with Apple MFi certification, braided and multi-color (ID: B0BPLX388R).\n- 2-pack 4ft Multi Fast Charging Cable 3 in 1 (Type C/Lightning/Micro USB) with 4.5A output (ID: B09X73T9DQ).\n- 4 in 1 U

In [49]:
print(output["raw_response"].usage)

CompletionUsage(completion_tokens=218, prompt_tokens=2761, total_tokens=2979, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=2688))


In [ ]:
rag_response = output[""]

print(rag_response.answer)

Yes, you can get charging cords. Here are some options:

1. 5pack iPhone Charger Lightning Cables (ID: B0BPLX388R) - Apple MFi certified, braided, multi-color, lengths 3, 6, and 10 ft.
2. Multi Fast Charging Cable 3 in 1 (ID: B09X73T9DQ) - Retractable, supports Lightning, Type C, Micro USB, 4.5A output.
3. SHEZI 4 in 1 USB C Cable Lightning Cable (ID: B0CCNXLP57) - 3ft, fast charging 45W, multi-port.
4. 2 Pack PD 20W USB C iPhone Charger with 10FT Cable (ID: B0CFG7V2QX) - Fast charging, MFi certified.
5. GREPHONE 2 Pack USB C to Lightning Cable, 6 FT (ID: B0BV6PWVCG) - MFi certified, fast charging.

Let me know if you want details or to purchase any of these.
